# Лекција 02 - Истраживање Microsoft Agent Framework-а

**Microsoft Agent Framework (MAF)** је јединствени оквир за изградњу AI агената. Пружа чисту, саставну архитектуру са четири основне грађевинске блокове:

- **Клијент** – повезује се са крајњом тачком AI модела и управља комуникацијом
- **Агент** – обавија клијента упутствима и дефиницијама алата
- **Алати** – проширују могућности агента прилагођеним функцијама које модел може позвати
- **Сесија** – одржава историју разговорa за интеракције са више корака

У овој лекцији ћемо изградити **агента за резервацију путовања** који проверава доступност дестинације користећи ове појмове.


## Подешавање


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Разумевање архитектуре Agent Framework-а

Microsoft Agent Framework прати слојевиту архитектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клијент** – `FoundryChatClient` се повезује са Azure OpenAI имплементацијом. Он обрађује аутентификацију, форматирање захтева и парсирање одговора.
2. **Агент** – Креиран из клијента путем `provider.create_agent()`, агент комбинује приступ моделу са инструкцијама (системски упит) и алатима.
3. **Алатке** – Python функције украшене са `@tool` које агент може позвати да изврши радње или преузме податке.
4. **Сесија** – Објекат `AgentSession` (направљен преко `agent.create_session()`) који чува историју разговора, омогућавајући дијалог са више корака где агент памти претходни контекст.

Хајде да изградимо сваки слој корак по корак.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Додавање алата са @tool декоратером

Алати омогућавају агентима да предузимају радње које превазилазе генерисање текста. `@tool` декоратер претвара обичну Python функцију у нешто што агент може да позове.

Кључне тачке:
- Користите `Annotated[type, "description"]` да би модел разумео сваки параметар.
- Докстринг постаје опис алата који модел види.
- `approval_mode="never_require"` значи да алат ради аутоматски без потврде корисника.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Креирање агента са алаткама

Сада комбинујемо клијента, инструкције и алатке у агента. `instructions` служе као системски упит — оне дефинишу личност и понашање агента.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Вишеокретни разговори са сесијама

Једна `AgentSession` (направљена преко `agent.create_session()`) бележи све поруке у разговору. Преношењем исте сесије у сваки позив `agent.run()`, агент има приступ целој историји разговора и може да се позове на раније поруке.

Прослеђујемо `tools=[check_destination_availability]` како би агент могао да позове наш проверавач доступности током сваког круга.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Резиме

У овој лекцији истражили сте четири стуба Microsoft Agent Framework-а:

| Концепт | Шта сте научили |
|---------|------------------|
| **Клијент** | `FoundryChatClient` се повезује са Azure OpenAI уз аутентификацију засновану на креденцијалима |
| **Агент** | `provider.create_agent()` повезује модел са упутствима и именом |
| **Алатке** | `@tool` декоратор изложује Python функције које агент може да позива |
| **Сесија** | `agent.create_session()` одржава историју разговора кроз више корака |

Ове грађевинске јединице се комбинују да би створиле агенте који могу да воде природне разговоре, позивају екстерне функције и одржавају контекст — темељ за напредније агентске шеме у каснијим лекцијама.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
